In [1]:
import gym
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.kernel_approximation import Nystroem,RBFSampler

In [2]:
GAMMA=0.9
ALPHA=.1

In [3]:
class Model:
    def __init__ (self,env):
        self.env=env
        states= get_all_states(env)
        
    def set_state(self,s):
        self.i1=s[0]
        self.i2=s[1]
        self.i3=s[2]
        self.i4=s[3]
        
   

In [4]:
def epsilon_greedy(env,policy,s,eps=.1):
    p=np.random.random()
    if p<(1-eps):
        return policy[s]
    else:
        return env.action_space.sample()

In [5]:
def get_all_states(env):
    all_states=[]
    s=env.reset()[0].tolist()
    all_states.append((s))
    for a in range(env.action_space.n):
        done=False
        while not done:
            s,r,done,info,i=env.step(a)
            if s.tolist() not in all_states:
                s=s.tolist()
                all_states.append((s))
    return all_states
        
    

In [11]:
def euclidean(v1, v2):
    return sum((p-q)**2 for p, q in zip(v1, v2)) ** .5

def nearest_state(s,all_states):
    foo = [euclidean(s, j) for j in all_states]
    i=np.argmin(foo)
    
    return all_states[i]
    
    
    
    

In [12]:
def play_game(env,policy):
    s=env.reset()[0].tolist()
    s=tuple(nearest_state(s,all_states))
    a=epsilon_greedy(env,policy,s)
    states=[s]
    actions=[a]
    rewards=[0]
    
    done=False
    
    while not done:
        s2,r,done,info,i=env.step(a)
        s2=tuple(nearest_state(s2,all_states))
        rewards.append(r)
        states.append(s2)
        a=epsilon_greedy(env,policy,s2)
        actions.append(a)    
    return states,actions,rewards

In [8]:
def max_dict(d):
    max_val=max(d.values())
    
    max_keys=[key for key , val in d.items() if val== max_val]
    
    return np.random.choice(max_keys) , max_val

In [22]:
if __name__=='__main__':
    env=gym.make("CartPole-v1", render_mode="rgb_array")
    
    all_states=[]
    for i in range (100):
        all_state=get_all_states(env)
        all_states=all_states+all_state
    print(len(all_states))
        
    
    
    policy={}
    for s in all_states:
        policy[tuple(s)]=env.action_space.sample()
        
            
    Q={}
    sample_counts={}
    state_sample_count={}
    for s in all_states:
        Q[tuple(s)]={}
        sample_counts[tuple(s)]={}
        state_sample_count[tuple(s)]=0
        for a in range(env.action_space.n):
            Q[tuple(s)][a]=0
            sample_counts[tuple(s)][a]=0
                    
     
    deltas=[]
    for it in range(10000):
        if it %1000==0:
            print(it)    
        biggest_change=0   
    
        states,actions,rewards=play_game(env,policy)
        states_actions=list(zip(states,actions))
        T = len(states)
        G=0
        for t in range(T-2,-1, -1):
            s=states[t]
            a=actions[t]           
            G=rewards[t+1] +GAMMA*G
        
            if (s,a) not in states_actions[:t]:
                old_q=Q[s][a]
                sample_counts[s][a] +=1
                lr =1/sample_counts[s][a]
                Q[s][a]=old_q +lr*(G-old_q) 
            
                policy[s]=max_dict(Q[s])[0]
                
                state_sample_count[s] +=1
                
                biggest_change=max(biggest_change,np.abs(old_q-Q[s][a]))
        
        deltas.append(biggest_change)
        
    plt.plot(deltas)
    plt.show()
     
        
        
    
    
    
    
                
            
    

c:\Users\4623\AppData\Local\Programs\Python\Python310\lib\site-packages\gym\envs\classic_control\cartpole.py:177: UserWarning: WARN: You are calling 'step()' even though this environment has already returned terminated = True. You should always call 'reset()' once you receive 'terminated = True' -- any further steps are undefined behavior.
  logger.warn(


1141
0
1000
2000
3000
4000
5000
6000
7000
